# Tokenizer Comparison: RoBERTa vs Qwen3 on Drilling Text

This notebook samples 100 unique OPERATION sentences from the drilling data
and visualizes how RoBERTa and Qwen3 tokenizers split them.

Key questions:
- How are drilling abbreviations (POOH, RIH, BHA, RSS) tokenized?
- How are numbers and units handled?
- Which tokenizer produces fewer / more meaningful tokens?

In [1]:
import pandas as pd
import glob
import os
import random
random.seed(42)

## 1. Sample 100 Unique OPERATION Sentences

In [2]:
data_dir = "/home/yahia.shaaban/project/drilling data/train"
parquet_files = sorted(glob.glob(os.path.join(data_dir, "*.parquet")))
print(f"Total parquet files: {len(parquet_files)}")

# Collect unique operations from a sample of files
all_operations = set()
for fpath in parquet_files[:50]:  # sample from 50 files for speed
    df = pd.read_parquet(fpath, columns=["OPERATION"])
    ops = df["OPERATION"].dropna().unique()
    all_operations.update(str(o) for o in ops)

print(f"Unique operations found: {len(all_operations)}")

# Sample 100
operations = sorted(all_operations)
sampled = random.sample(operations, min(100, len(operations)))
print(f"Sampled {len(sampled)} sentences")
print()
print("--- First 10 samples ---")
for s in sampled[:10]:
    print(f"  {s[:120]}")

Total parquet files: 1738
Unique operations found: 7318
Sampled 100 sentences

--- First 10 samples ---
  PROGRAMMED MWD TOOLS & DATA
  CONT  TDS BENTEC HEALTHY INSPECTION
  BREAK OUT & L/D 9 5/8" CSG L/J, CRT TOOL, R/D 9-5/8" CSG RUNNING EQUIP.
  RIH 3-1/2" - VAMTOP 9.2# ALLOY 28 UPPER COMPLETION  (as per Tally) T/6,911'  
  DRILL 6" H.HOLE W/ BAKER RSS F/13,271' T/13,473' - 280 GPM/2400 PSI, 120 RPM, 5-10 KFTLB, 10-15 KLBS.
  CONTINUE RIH 4-1/2" TSH BLUE L-80 LEL TBG F/1369 FT TO 1933 FT. P/UP AND  M/UP  WFORD WATER SWELL PKR # 2 ASSY AND RIH S
  CONT. DRILLING 12¼" DEV HOLE W SLB RSS BHA FROM 3347' to 3452' [R-2]. OBSERVED VERY LOW ROP
  CONT POOH W/ HLB 8 1/2' RSS/MWD BHA F/ 4253' TO 820'
  RIH 2-7/8'' TBG CEMENT STRING FROM 1169' TO 2520' WASH DOWN LAST STDS. TAG TOC @ 2685FT. 
  CONDUCTED PJSM PRIOR DISPLACEING WELL TO INHIBITED/FILTRATED 10.7 PPG BRINE.


## 2. Load Tokenizers

In [3]:
from transformers import AutoTokenizer

roberta_tok = AutoTokenizer.from_pretrained("roberta-base")
qwen3_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B", trust_remote_code=True)

print(f"RoBERTa vocab size: {roberta_tok.vocab_size:,}")
print(f"Qwen3 vocab size:   {qwen3_tok.vocab_size:,}")

/home/yahia.shaaban/anaconda3/envs/tslm/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RoBERTa vocab size: 50,265
Qwen3 vocab size:   151,643


## 3. Tokenization Comparison Table

For each sentence, show: original text, RoBERTa tokens, Qwen3 tokens, token counts.

In [4]:
from IPython.display import HTML

def colorize_tokens(tokens, tokenizer_name):
    """Color tokens alternating for readability."""
    colors = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6",
              "#1abc9c", "#e67e22", "#3498db", "#e74c3c", "#2ecc71"]
    html_parts = []
    for i, tok in enumerate(tokens):
        color = colors[i % len(colors)]
        escaped = tok.replace('<', '&lt;').replace('>', '&gt;').replace(' ', '&nbsp;')
        html_parts.append(f'<span style="background-color:{color}22;border:1px solid {color};'
                          f'border-radius:3px;padding:1px 3px;margin:1px;'
                          f'font-family:monospace;font-size:12px;color:{color}">{escaped}</span>')
    return ' '.join(html_parts)

# Build comparison data
rows = []
for sent in sampled:
    r_ids = roberta_tok.encode(sent, add_special_tokens=False)
    q_ids = qwen3_tok.encode(sent, add_special_tokens=False)
    r_tokens = roberta_tok.convert_ids_to_tokens(r_ids)
    q_tokens = qwen3_tok.convert_ids_to_tokens(q_ids)
    rows.append({
        'text': sent,
        'roberta_tokens': r_tokens,
        'qwen3_tokens': q_tokens,
        'roberta_count': len(r_tokens),
        'qwen3_count': len(q_tokens),
    })

df_comp = pd.DataFrame(rows)
print(f"Average token count — RoBERTa: {df_comp['roberta_count'].mean():.1f}, Qwen3: {df_comp['qwen3_count'].mean():.1f}")
print(f"Max token count    — RoBERTa: {df_comp['roberta_count'].max()}, Qwen3: {df_comp['qwen3_count'].max()}")

Token indices sequence length is longer than the specified maximum sequence length for this model (568 > 512). Running this sequence through the model will result in indexing errors


Average token count — RoBERTa: 38.5, Qwen3: 39.4
Max token count    — RoBERTa: 568, Qwen3: 507


In [5]:
# Display first 30 with colored tokens
html = '<table style="border-collapse:collapse;width:100%">'
html += '<tr style="background:#f0f0f0">'
html += '<th style="padding:8px;text-align:left;width:5%">#</th>'
html += '<th style="padding:8px;text-align:left;width:30%">Original Text</th>'
html += '<th style="padding:8px;text-align:left;width:30%">RoBERTa Tokens</th>'
html += '<th style="padding:8px;text-align:center;width:3%">N</th>'
html += '<th style="padding:8px;text-align:left;width:30%">Qwen3 Tokens</th>'
html += '<th style="padding:8px;text-align:center;width:3%">N</th>'
html += '</tr>'

for i, row in df_comp.head(30).iterrows():
    bg = '#ffffff' if i % 2 == 0 else '#f9f9f9'
    text_short = row['text'][:100] + ('...' if len(row['text']) > 100 else '')
    html += f'<tr style="background:{bg}">'
    html += f'<td style="padding:6px;font-size:12px">{i+1}</td>'
    html += f'<td style="padding:6px;font-size:11px;font-family:monospace">{text_short}</td>'
    html += f'<td style="padding:6px">{colorize_tokens(row["roberta_tokens"][:25], "roberta")}</td>'
    html += f'<td style="padding:6px;text-align:center;font-weight:bold">{row["roberta_count"]}</td>'
    html += f'<td style="padding:6px">{colorize_tokens(row["qwen3_tokens"][:25], "qwen3")}</td>'
    html += f'<td style="padding:6px;text-align:center;font-weight:bold">{row["qwen3_count"]}</td>'
    html += '</tr>'

html += '</table>'
HTML(html)

## 4. Focus: Drilling Abbreviations

How do the tokenizers handle key drilling terms?

In [ ]:
drilling_terms = [
    "POOH", "RIH", "BHA", "RSS", "MWD", "LWD", "TDS", "HWDP",
    "CMT", "CSG", "BOP", "NRV", "TBG", "PKR", "CHH", "PPG",
    "PJSM", "TOFSM", "DRLOUT", "WLHD", "SRFEQ",
    "N/U", "R/U", "L/D", "P/T", "M/U", "R/D", "W/", "F/",
    "CONT'D", "CONT'D DRLG",
    # Numbers + units
    "2500 PSI", "12 1/4\"", "8.5\"", "59 FT/HR", "7710 FT",
    "1.12 SG", "10 BPH", "420 GPM",
    # Full phrases
    "POOH W/ 8.5\" RSS BHA F/ 12030' TO 9330'",
    "RIH W/ 12 1/4\" BIT + MWD/LWD BHA",
    "P/T SURFACE LINES TO 2500 PSI - 10 MIN - OK",
    "DRILLED 12 1/4\" V.HOLE W/ IDS RSS/MWD BHA F/ 2180' TO 2845'",
]

html = '<table style="border-collapse:collapse;width:100%">'
html += '<tr style="background:#2c3e50;color:white">'
html += '<th style="padding:8px;text-align:left">Drilling Term</th>'
html += '<th style="padding:8px;text-align:left">RoBERTa Tokens</th>'
html += '<th style="padding:8px;text-align:center">N</th>'
html += '<th style="padding:8px;text-align:left">Qwen3 Tokens</th>'
html += '<th style="padding:8px;text-align:center">N</th>'
html += '</tr>'

for i, term in enumerate(drilling_terms):
    r_ids = roberta_tok.encode(term, add_special_tokens=False)
    q_ids = qwen3_tok.encode(term, add_special_tokens=False)
    r_toks = roberta_tok.convert_ids_to_tokens(r_ids)
    q_toks = qwen3_tok.convert_ids_to_tokens(q_ids)
    
    bg = '#ffffff' if i % 2 == 0 else '#f9f9f9'
    html += f'<tr style="background:{bg}">'
    html += f'<td style="padding:6px;font-family:monospace;font-weight:bold">{term}</td>'
    html += f'<td style="padding:6px">{colorize_tokens(r_toks, "roberta")}</td>'
    html += f'<td style="padding:6px;text-align:center;font-weight:bold">{len(r_toks)}</td>'
    html += f'<td style="padding:6px">{colorize_tokens(q_toks, "qwen3")}</td>'
    html += f'<td style="padding:6px;text-align:center;font-weight:bold">{len(q_toks)}</td>'
    html += '</tr>'

html += '</table>'
HTML(html)

## 5. Token Count Distribution

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of token counts
axes[0].hist(df_comp['roberta_count'], bins=20, alpha=0.7, label='RoBERTa', color='#3498db')
axes[0].hist(df_comp['qwen3_count'], bins=20, alpha=0.7, label='Qwen3', color='#e74c3c')
axes[0].set_xlabel('Token Count per Sentence')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Token Count Distribution (100 Drilling Sentences)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter: RoBERTa vs Qwen3 token count
axes[1].scatter(df_comp['roberta_count'], df_comp['qwen3_count'], 
                alpha=0.6, c='#2ecc71', edgecolors='#27ae60', s=40)
max_val = max(df_comp['roberta_count'].max(), df_comp['qwen3_count'].max()) + 5
axes[1].plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Equal')
axes[1].set_xlabel('RoBERTa Token Count')
axes[1].set_ylabel('Qwen3 Token Count')
axes[1].set_title('Token Count: RoBERTa vs Qwen3')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('tokenizer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Stats
print(f"\nRoBERTa — mean: {df_comp['roberta_count'].mean():.1f}, median: {df_comp['roberta_count'].median():.0f}")
print(f"Qwen3   — mean: {df_comp['qwen3_count'].mean():.1f}, median: {df_comp['qwen3_count'].median():.0f}")
print(f"\nRoBERTa uses {df_comp['roberta_count'].mean() / df_comp['qwen3_count'].mean():.2f}x more tokens on average")

## 6. Worst Cases: Where Tokenization Breaks Down

In [6]:
# Find sentences where RoBERTa produces the most tokens (worst fragmentation)
df_comp['ratio'] = df_comp['roberta_count'] / df_comp['qwen3_count'].clip(lower=1)

print("=" * 80)
print("  WORST FRAGMENTATION (RoBERTa produces most extra tokens vs Qwen3)")
print("=" * 80)
for _, row in df_comp.nlargest(10, 'ratio').iterrows():
    print(f"\n  Text: {row['text'][:100]}")
    print(f"  RoBERTa: {row['roberta_count']} tokens -> {row['roberta_tokens'][:15]}")
    print(f"  Qwen3:   {row['qwen3_count']} tokens -> {row['qwen3_tokens'][:15]}")
    print(f"  Ratio:   {row['ratio']:.2f}x")

  WORST FRAGMENTATION (RoBERTa produces most extra tokens vs Qwen3)

  Text: PJSM  
  RoBERTa: 5 tokens -> ['P', 'J', 'SM', 'Ġ', 'Ġ']
  Qwen3:   3 tokens -> ['PJ', 'SM', 'ĠĠ']
  Ratio:   1.67x

  Text: RECIEVED NEW MOTOR. INSTALL MOTOR .
  RoBERTa: 13 tokens -> ['REC', 'IE', 'V', 'ED', 'ĠNEW', 'ĠMOT', 'OR', '.', 'ĠINST', 'ALL', 'ĠMOT', 'OR', 'Ġ.']
  Qwen3:   9 tokens -> ['REC', 'IE', 'VED', 'ĠNEW', 'ĠMOTOR', '.', 'ĠINSTALL', 'ĠMOTOR', 'Ġ.']
  Ratio:   1.44x

  Text: R/D BAKER TRS-PC MACHINE
  RoBERTa: 13 tokens -> ['R', '/', 'D', 'ĠB', 'AK', 'ER', 'ĠT', 'RS', '-', 'PC', 'ĠM', 'ACH', 'INE']
  Qwen3:   9 tokens -> ['R', '/D', 'ĠB', 'AKER', 'ĠTR', 'S', '-', 'PC', 'ĠMACHINE']
  Ratio:   1.44x

  Text: CIRCULATE  CONDITION HOLE
  RoBERTa: 10 tokens -> ['C', 'IRC', 'UL', 'ATE', 'Ġ', 'ĠCON', 'D', 'ITION', 'ĠHO', 'LE']
  Qwen3:   7 tokens -> ['C', 'IRC', 'ULATE', 'Ġ', 'ĠCONDITION', 'ĠH', 'OLE']
  Ratio:   1.43x

  Text: CONDUCTED KICK DRILL WHILE TRIPPING. GOOD RESPONSE.
  RoBERTa: 20 tokens -

## 7. Single Token Analysis: Which Drilling Terms Are Single Tokens?

In [ ]:
key_terms = [
    "POOH", "RIH", "BHA", "RSS", "MWD", "LWD", "TDS", "HWDP",
    "CMT", "CSG", "BOP", "NRV", "TBG", "PKR", "PPG", "BPH",
    "PJSM", "DRLOUT", "WLHD", "SRFEQ", "CIRC", "REAM",
    "DRILL", "TRIP", "SAFE", "WAIT", "CORE", "FISH",
]

print(f"{'Term':<10} {'RoBERTa':>10} {'Qwen3':>10}  RoBERTa Tokens                Qwen3 Tokens")
print("-" * 90)

for term in key_terms:
    r_ids = roberta_tok.encode(term, add_special_tokens=False)
    q_ids = qwen3_tok.encode(term, add_special_tokens=False)
    r_toks = roberta_tok.convert_ids_to_tokens(r_ids)
    q_toks = qwen3_tok.convert_ids_to_tokens(q_ids)
    
    r_mark = 'ok' if len(r_toks) == 1 else f'{len(r_toks)} parts'
    q_mark = 'ok' if len(q_toks) == 1 else f'{len(q_toks)} parts'
    
    print(f"{term:<10} {r_mark:>10} {q_mark:>10}  {str(r_toks):<30s} {str(q_toks)}")

## Summary

Key takeaways:
1. **Fragmentation**: How many drilling terms get split into meaningless subwords?
2. **Vocabulary overlap**: Which tokenizer already knows more drilling terms?
3. **Number handling**: How are depths, pressures, and rates tokenized?
4. **Decision**: Based on this analysis, decide whether to:
   - Extend RoBERTa's tokenizer with drilling terms + continue pretraining
   - Use Qwen3 as-is (larger vocab, possibly better coverage)
   - Train a domain-specific tokenizer from scratch